# Descriptive-Title
Felix Zaussinger | XX.YY.ZZZZ

## Core Analysis Goal(s)
1.
2.
3.

## Key Insight(s)
1.
2.
3.

In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

Initialise ESCO class

In [2]:
from src.data.framework import Esco
esco = Esco()

ESCO occupations are arranged in a nested hierarchical structure.

They extend 4-digit ISCO occupations to a 5th, 6th, 7th and 8th level.

The code variable includes the information about the ESCO occupation hierarchy level.

In [100]:
esco.occupations

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,technical and operations director\nhead of tec...,NaN,released,2016-07-05T13:58:41Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Technical directors realise the artistic visio...,2654.1.7
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,metal drawing machine technician\nmetal drawin...,NaN,released,2016-07-05T17:09:43Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,inspector of precision instruments\nprecision ...,NaN,released,2016-07-06T09:21:20Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Precision device inspectors make sure precisio...,7543.10.3
3,Occupation,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,air traffic safety electronics hardware specia...,NaN,released,2017-01-17T11:40:37Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Air traffic safety technicians provide technic...,3155.1
4,Occupation,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,hospitality revenues manager\nyield manager\nh...,NaN,released,2017-01-17T13:33:42Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Hospitality revenue managers maximise revenue ...,2431.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3003,Occupation,http://data.europa.eu/esco/occupation/ff656b3a...,2120,demographer,demography research analyst\ndemography studie...,NaN,released,2021-12-21T13:41:38.024Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Demographers study a variety of parameters rel...,2120.2
3004,Occupation,http://data.europa.eu/esco/occupation/ff8d4065...,9612,sorter labourer,grader\nyard labourer\nrecycler\nrecycling sit...,NaN,released,2021-12-08T20:18:37.546Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Sorter labourers sort recyclable materials and...,9612.2
3005,Occupation,http://data.europa.eu/esco/occupation/ffa4dd5d...,5414,armoured car guard,armoured truck escort\ntruck escort\narmoured ...,NaN,released,2022-01-13T09:51:50.643Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Armoured car guards ensure the safe transporta...,5414.1.2
3006,Occupation,http://data.europa.eu/esco/occupation/ffade2f4...,2422,civil service administrative officer,government administrative officer\nlocal autho...,NaN,released,2016-07-05T16:17:26Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Civil service administrative officers perform ...,2422.1


We can thus retrieve the level for each occupation and build identifiers for ESCO levels 5-8.

In [21]:
split_codes = esco.occupations.code.str.split(".", expand=True)
split_codes

,0,1,2,3,4
0,2654,1,7,None,None
1,8121,4,None,None,None
2,7543,10,3,None,None
3,3155,1,None,None,None
4,2431,9,None,None,None
...,...,...,...,...,...
3003,2120,2,None,None,None
3004,9612,2,None,None,None
3005,5414,1,2,None,None
3006,2422,1,None,None,None


In [92]:
occ_hierarchy = esco.occupations.copy()

# derive isco level codes
occ_hierarchy["isco_lvl_1"] = occ_hierarchy["iscoGroup"].str[0]
occ_hierarchy["isco_lvl_2"] = occ_hierarchy["iscoGroup"].str[:2]
occ_hierarchy["isco_lvl_3"] = occ_hierarchy["iscoGroup"].str[:3]
occ_hierarchy["isco_lvl_4"] = occ_hierarchy["iscoGroup"].str[:4]

# Version 1: split codes at dots
occ_hierarchy["esco_lvl_5"] = split_codes.iloc[:, 0] + "." + split_codes.iloc[:, 1]
occ_hierarchy["esco_lvl_6"] = split_codes.iloc[:, 0] + "." + split_codes.iloc[:, 1] + "." + split_codes.iloc[:, 2]
occ_hierarchy["esco_lvl_7"] = split_codes.iloc[:, 0] + "." + split_codes.iloc[:, 1] + "." + split_codes.iloc[:, 2] + "." + split_codes.iloc[:, 3]
occ_hierarchy["esco_lvl_8"] = split_codes.iloc[:, 0] + "." + split_codes.iloc[:, 1] + "." + split_codes.iloc[:, 2] + "." + split_codes.iloc[:, 3] + "." + split_codes.iloc[:, 4]

# Version 2: specify relationship to hierarchy levels through (non-) existence of a deeper/more granular level
occ_hierarchy["is_esco_lvl_5"] = split_codes.iloc[:, 2].isna()
occ_hierarchy["is_esco_lvl_6"] = split_codes.iloc[:, 2].notna() & split_codes.iloc[:, 3].isna()
occ_hierarchy["is_esco_lvl_7"] = split_codes.iloc[:, 2].notna() & split_codes.iloc[:, 3].notna() & split_codes.iloc[:, 4].isna()
occ_hierarchy["is_esco_lvl_8"] = split_codes.iloc[:, 2].notna() & split_codes.iloc[:, 3].notna() & split_codes.iloc[:, 4].notna()

In [99]:
occ_hierarchy.describe()

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,isco_lvl_1,isco_lvl_2,isco_lvl_3,isco_lvl_4,esco_lvl_5,esco_lvl_6,esco_lvl_7,esco_lvl_8,is_esco_lvl_5,is_esco_lvl_6,is_esco_lvl_7,is_esco_lvl_8
count,3008,3008,3008,3008,2974,6,3008,3008,3008,304,8,3008,3008,3008,3008,3008,3008,3008,3008,1252,175,40,3008,3008,3008,3008
unique,1,3008,426,3008,2974,6,1,2940,2,272,8,2,3008,3008,10,42,125,426,1756,1077,135,40,2,2,2,2
top,Occupation,http://data.europa.eu/esco/occupation/00030d09...,1324,technical director,technical and operations director\nhead of tec...,youth policy manager\nyouth planner\nyouth coa...,released,2021-12-13T21:32:11.472Z,http://data.europa.eu/esco/regulated-professio...,Excludes people performing managerial activities.,Excludes choreologist.,http://data.europa.eu/esco/concept-scheme/occu...,Technical directors realise the artistic visio...,2654.1.7,2,21,311,1324,1324.3,1324.3.1,1324.3.1.6,1324.3.1.6.11,True,False,False,False
freq,3008,1,90,1,1,1,3008,13,2993,12,1,1558,1,1,846,272,134,90,79,39,33,1,1756,1931,2873,2968


In [101]:
occ_hierarchy

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,isco_lvl_1,isco_lvl_2,isco_lvl_3,isco_lvl_4,esco_lvl_5,esco_lvl_6,esco_lvl_7,esco_lvl_8,is_esco_lvl_5,is_esco_lvl_6,is_esco_lvl_7,is_esco_lvl_8
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,technical and operations director\nhead of tec...,NaN,released,2016-07-05T13:58:41Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Technical directors realise the artistic visio...,2654.1.7,2,26,265,2654,2654.1,2654.1.7,NaN,NaN,False,True,False,False
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,metal drawing machine technician\nmetal drawin...,NaN,released,2016-07-05T17:09:43Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4,8,81,812,8121,8121.4,NaN,NaN,NaN,True,False,False,False
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,inspector of precision instruments\nprecision ...,NaN,released,2016-07-06T09:21:20Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Precision device inspectors make sure precisio...,7543.10.3,7,75,754,7543,7543.10,7543.10.3,NaN,NaN,False,True,False,False
3,Occupation,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,air traffic safety electronics hardware specia...,NaN,released,2017-01-17T11:40:37Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Air traffic safety technicians provide technic...,3155.1,3,31,315,3155,3155.1,NaN,NaN,NaN,True,False,False,False
4,Occupation,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,hospitality revenues manager\nyield manager\nh...,NaN,released,2017-01-17T13:33:42Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Hospitality revenue managers maximise revenue ...,2431.9,2,24,243,2431,2431.9,NaN,NaN,NaN,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3003,Occupation,http://data.europa.eu/esco/occupation/ff656b3a...,2120,demographer,demography research analyst\ndemography studie...,NaN,released,2021-12-21T13:41:38.024Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Demographers study a variety of parameters rel...,2120.2,2,21,212,2120,2120.2,NaN,NaN,NaN,True,False,False,False
3004,Occupation,http://data.europa.eu/esco/occupation/ff8d4065...,9612,sorter labourer,grader\nyard labourer\nrecycler\nrecycling sit...,NaN,released,2021-12-08T20:18:37.546Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Sorter labourers sort recyclable materials and...,9612.2,9,96,961,9612,9612.2,NaN,NaN,NaN,True,False,False,False
3005,Occupation,http://data.europa.eu/esco/occupation/ffa4dd5d...,5414,armoured car guard,armoured truck escort\ntruck escort\narmoured ...,NaN,released,2022-01-13T09:51:50.643Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Armoured car guards ensure the safe transporta...,5414.1.2,5,54,541,5414,5414.1,5414.1.2,NaN,NaN,False,True,False,False
3006,Occupation,http://data.europa.eu/esco/occupation/ffade2f4...,2422,civil service administrative officer,government administrative officer\nlocal autho...,NaN,released,2016-07-05T16:17:26Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Civil service administrative officers perform ...,2422.1,2,24,242,2422,242

Save occupation hierarchy metadata

In [94]:
utils.save_df_to_files(
    df=occ_hierarchy,
    output_dir=os.path.join(useful_paths.data_processed, "esco"),
    fname_no_ext="esco_occupation_hierarchy"
)